# WAKIS: Rechteckresonator mit einer SIBC-Wand

**Ziel:** PEC-Referenzfrequenz des $\mathrm{TE}_{101}$-Modes und erste quantitative Validierung der SIBC-Wandverluste. Fünf Wände sind äußere PEC-Randbedingungen; die sechste Wand (bei $x=x_w$) ist die ebene Grenzfläche eines STL-Blocks. Für beide Varianten wird **dieselbe STL-Geometrie** genutzt, mit Materialtyp `pec` bzw. `sibc`.

Mit `conformal_pec_snap_fraction=0.2` werden kleine aktive Schnittanteile an der Leitergrenze gesnappt. **PEC und SIBC verwenden dieselbe Snap-Fraction.** Die Wand ist bewusst leicht gegenüber dem kartesischen Gitter versetzt, damit echte konforme Schnittflächen entstehen. Die geometrisch aktive Kavität hat $a=L_x-x_w$, $b=L_y$, $d=L_z$. Die äußere x-Min-Randbedingung liegt *hinter* dem Leiter. Für die erste Validierung wird die Eigenmode mit der analytischen TE101-Frequenz verglichen und der Wandverlust über eine Störung erster Ordnung ermittelt. Das ist zuverlässiger als ein grober Frequenzsweep bei hohem Q.

**Voraussetzungen:** Aktuelle `SolverFIT3D`-Materialkorrektur und `SolverFIT3D_FD` mit konsequentem `order="F"` sowie aktiver $D_{Z_s}$-Korrektur. Das Notebook wurde nicht in deiner vollständigen WAKIS-Laufzeit ausgeführt.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import numpy as np
import pyvista as pv
from scipy.constants import c, mu_0, pi
from wakis import GridFIT3D
from wakis.solverFIT3D_FD import SolverFIT3D_FD

# Geometry: all values in meters
Lx, Ly, Lz = 0.36, 0.22, 0.26
x_wall = 0.087  # not on a mesh plane (dx = 0.02 m)
Nx, Ny, Nz = 18, 11, 13
eta = 0.2  # PEC/SIBC conformal snapping threshold
sigma_wall = 5.8e7  # copper-like conductivity [S/m]

# Inner rectangular vacuum cavity: x_wall < x < Lx
# TE101: E_y ~ sin(pi*(x-x_wall)/a) sin(pi*z/d)
a, b, d = Lx - x_wall, Ly, Lz
kx, kz = pi / a, pi / d
f_analytical = c / 2 * np.sqrt((1 / a) ** 2 + (1 / d) ** 2)
print(f"TE101 analytical PEC frequency: {f_analytical / 1e9:.6f} GHz")

TE101 analytical PEC frequency: 0.796152 GHz


In [2]:
tmp = TemporaryDirectory()
stl_path = Path(tmp.name) / "left_wall.stl"
# A single closed box extends beyond the x-low, y and z simulation boundaries.
# Inside the domain, only its right face x=x_wall is physically exposed.
solid = pv.Cube(
    center=((x_wall - 0.12) / 2, Ly / 2, Lz / 2),
    x_length=x_wall + 0.12,
    y_length=Ly + 0.20,
    z_length=Lz + 0.20,
).triangulate()
solid.save(str(stl_path))


def make_solver(material_type):
    mat = [np.inf, 1.0, 0.0] if material_type == "pec" else [1.0, 1.0, sigma_wall]
    grid = GridFIT3D(
        xmin=0.0,
        xmax=Lx,
        ymin=0.0,
        ymax=Ly,
        zmin=0.0,
        zmax=Lz,
        Nx=Nx,
        Ny=Ny,
        Nz=Nz,
        stl_solids={"wall": str(stl_path)},
        stl_materials={"wall": mat},
        stl_material_types={"wall": material_type},
        geometry_mode="conformal",
        stl_method="interior_points",
        bg_material_type="normal",
        conformal_pec_snap_fraction=eta,
        verbose=0,
    )
    return SolverFIT3D_FD(
        grid=grid,
        use_stl=True,
        use_sibc=False,
        bc_low=["PEC"] * 3,
        bc_high=["PEC"] * 3,
        verbose=0,
    )


pec = make_solver("pec")
sibc = make_solver("sibc")
print("E DOFs (PEC/SIBC):", len(pec.free_E_idx), len(sibc.free_E_idx))
assert np.array_equal(pec.free_E_idx, sibc.free_E_idx), "PEC/SIBC DOF mismatch"
print("SIBC face coefficients:", np.count_nonzero(sibc.sibc_g > 0))
assert sibc.has_sibc

# Verify that the same snapping produces comparable reduced PEC/SIBC operators.
assert pec.C_red.shape == sibc.C_red.shape
assert np.allclose(pec.Meps.diagonal(), sibc.Meps.diagonal()), "PEC/SIBC Meps differ"
assert np.allclose(pec.Mnu.diagonal(), sibc.Mnu.diagonal()), "PEC/SIBC Mnu differ"
print(f"Conformal snap fraction: {eta:.2f}")

Generating grid with 2574 mesh cells...


/home/hschreib/Documents/Wakis/WAKIS/wakis/gridFIT3D.py:4428: UserWarning: Conformal geometry diagnostics:
168 edges contain geometric crossings that are hidden from endpoint topology and are ignored by the current conformal discretization.
  self._warn_interface_diagnostics()


Initializing Electromagnetic solver...
Generating grid with 2574 mesh cells...
Initializing Electromagnetic solver...
E DOFs (PEC/SIBC): 5720 5720
SIBC face coefficients: 286
Conformal snap fraction: 0.20


## 1. Geometrische und algebraische Kontrolle

Die SIBC-Flächen sollen bei diesem Aufbau überwiegend y- und z-normal sein, weil die ebene Leiteroberfläche senkrecht zur x-Achse liegt und entsprechende FIT-Flächen schneidet. Prüfe, ob relevante Korrekturen mit den aktiven elektrischen Freiheitsgraden gekoppelt sind.

In [3]:
g = sibc.sibc_g
active = g > 0
row_active = np.diff(sibc.C_red.tocsr().indptr) > 0
for comp, direction in enumerate("xyz"):
    m = active[comp * sibc.N : (comp + 1) * sibc.N]
    r = row_active[comp * sibc.N : (comp + 1) * sibc.N]
    print(f"{direction}: {m.sum()} SIBC faces, {(m & r).sum()} coupled")
print("Total active SIBC:", active.sum(), "coupled:", (active & row_active).sum())
print(
    "Material-only DOFs removed:",
    np.count_nonzero(
        sibc.active_conductor_E
        & sibc.bc_mask_E.toarray().astype(bool)
        & (sibc.iDeps.diagonal() == 0)
    ),
)

x: 0 SIBC faces, 0 coupled
y: 143 SIBC faces, 132 coupled
z: 143 SIBC faces, 132 coupled
Total active SIBC: 286 coupled: 264
Material-only DOFs removed: 0


## 2. PEC-Resonanzfrequenz

Wir bestimmen den PEC-Eigenmode in der Nähe von $f_\mathrm{TE101}$. Für das Finite-Grid-Ergebnis ist eine gewisse Diskretisierungsabweichung normal. Falls mehrere Moden nahe beieinanderliegen, muss die TE101-Mode anhand ihrer Feldverteilung identifiziert werden.

In [4]:
frequencies, modes, eigenvalues = pec.solve_eigenmodes(
    n_modes=6, target_frequency=f_analytical
)
print("Eigenfrequencies [GHz]:", np.round(frequencies / 1e9, 6))
valid = np.flatnonzero(np.isfinite(frequencies) & (frequencies > 0))
assert len(valid) > 0
k_mode = valid[np.argmin(np.abs(frequencies[valid] - f_analytical))]
f0 = float(frequencies[k_mode])
omega0 = 2 * pi * f0
e = modes[:, k_mode].astype(complex)
print(f"Selected mode {k_mode}: f_FIT = {f0 / 1e9:.6f} GHz")
print(f"Relative PEC frequency error = {(f0 / f_analytical - 1) * 100:.3f} %")

Eigenfrequencies [GHz]: [     nan 0.795779 0.87497  0.89169  1.046607 1.047743]
Selected mode 1: f_FIT = 0.795779 GHz
Relative PEC frequency error = -0.047 %


## 3. SIBC-Verlust und störungstheoretischer Gütefaktor

Wir benutzen die modebasierte Näherung (mit $\exp(+j\omega t)$):

$$\delta\omega \simeq \frac{\mathbf e^H[\mathbf S_{L,\mathrm{SIBC}}(\omega_0)-\mathbf S_{L,\mathrm{PEC}}]\mathbf e}{2\omega_0\,\mathbf e^H\mathbf M_\varepsilon\mathbf e},\qquad Q_{\rm FIT}\simeq\frac{\omega_0}{2\operatorname{Im}\delta\omega}. $$

Für einen passiven Leiter muss bei dieser Zeitkonvention $\operatorname{Im}\delta\omega>0$ sein. Die Näherung setzt geringe Verluste und eine ausreichend isolierte Mode voraus.

Der analytische Vergleich berücksichtigt **nur eine verlustbehaftete x-Wand**; die anderen fünf Wände bleiben ideal PEC. Mit $R_s=\sqrt{\omega\mu_0/(2\sigma)}$ gilt für TE101:

$$Q_{\mathrm{wall}} = \frac{\omega\mu_0}{R_s}\frac{a(k_x^2+k_z^2)}{2 k_x^2}. $$

Dies ist der Oberflächenverlust der analytischen Kavitätsmode, keine exakte Lösung des diskreten konformen Modells.

In [5]:
D = sibc._build_sibc_diagonal(omega0)
mnu = sibc.Mnu.diagonal()
C = sibc.C_red.tocsr()
from scipy.sparse import diags

K_sibc = (C.T @ diags(mnu / D) @ C).tocsr()
dK = K_sibc - pec.SL
numerator = np.vdot(e, dK @ e)
denominator = 2 * omega0 * np.vdot(e, pec.Meps @ e)
domega = numerator / denominator

Rs = np.sqrt(omega0 * mu_0 / (2 * sigma_wall))
Q_analytic = (omega0 * mu_0 / Rs) * a * (kx * kx + kz * kz) / (2 * kx * kx)

print(f"R_s(PEC reference frequency): {Rs:.6g} ohm")
print(f"Complex frequency shift δω/(2π): {domega / (2 * pi):.6g} Hz")
if domega.imag > 0:
    Q_fit = omega0 / (2 * domega.imag)
    print(f"Q_FIT (first-order): {Q_fit:.4g}")
    print(f"Q_analytic (one lossy wall): {Q_analytic:.4g}")
    print(f"Ratio Q_FIT/Q_analytic: {Q_fit / Q_analytic:.4g}")
else:
    print("WARNING: no positive damping. Check SIBC sign and operator!")
print(f"Real frequency shift: {domega.real / (2 * pi):+.6g} Hz")

R_s(PEC reference frequency): 0.00735973 ohm
Complex frequency shift δω/(2π): -1630.04+1629.75j Hz
Q_FIT (first-order): 2.441e+05
Q_analytic (one lossy wall): 2.45e+05
Ratio Q_FIT/Q_analytic: 0.9964
Real frequency shift: -1630.04 Hz


## 4. Leitfähigkeitsskalierung

Zur zusätzlichen Kontrolle berechnen wir $Q$ mit denselben Geometriefaktoren für mehrere Leitfähigkeiten. Erwartet wird bei gutem Leiter $Q\propto\sqrt{\sigma}$. Dieser Test prüft die Frequenz-/Leitfähigkeitsskalierung des **implementierten Operators**, noch nicht dessen absolute physikalische Richtigkeit.

In [6]:
sigma_ref = sibc.sibc_sigma.copy()
for sigma in (1e5, 1e6, 1e7, 5.8e7):
    try:
        sibc.sibc_sigma[active] = sigma
        Dtest = sibc._build_sibc_diagonal(omega0)
        DeltaK = (C.T @ diags(mnu * (1 / Dtest - 1)) @ C).tocsr()
        delta = np.vdot(e, DeltaK @ e) / denominator
        q = omega0 / (2 * delta.imag) if delta.imag > 0 else np.nan
        print(
            f"sigma={sigma:.2e} S/m, Q={q:.6g}, Q/sqrt(sigma)={q / np.sqrt(sigma):.6g}"
        )
    finally:
        sibc.sibc_sigma[:] = sigma_ref

sigma=1.00e+05 S/m, Q=10179.7, Q/sqrt(sigma)=32.1909
sigma=1.00e+06 S/m, Q=32095.6, Q/sqrt(sigma)=32.0956
sigma=1.00e+07 S/m, Q=101400, Q/sqrt(sigma)=32.0655
sigma=5.80e+07 S/m, Q=244141, Q/sqrt(sigma)=32.0573


## Interpretation und Grenzen

- Die Geometrie enthält eine ebene Leiterfläche, wodurch unerwünschte Mehrfachschnitte deutlich reduziert werden. Melden die Geometriediagnosen dennoch ungültige Flächen, sollte die Geometrie vor weiteren Tests angepasst werden.
- **Modeidentifikation:** Der nächstliegende Eigenwert ist nicht automatisch TE101; bei unerwarteten Ergebnissen die E-Feldstruktur prüfen.
- **Vorzeichen:** $\operatorname{Im}\delta\omega>0$ ist unter der verwendeten Zeitkonvention ein notwendiger Test für positive Dämpfung.
- **Absolute Genauigkeit:** Die erste SIBC-Version nutzt die skalare $\sin\theta$-Approximation. Eine Übereinstimmung im $\sqrt\sigma$-Gesetz ist noch kein Beleg dafür, dass Geometriefaktoren und Verluste quantitativ stimmen.
- Nach dieser schnellen Störungsanalyse können wir mit einem adaptiven FD-Frequenzsweep die Resonanzbreite unabhängig bestimmen. Für Kupfer ist ein Sweep über viele Prozent der Resonanzfrequenz normalerweise zu grob, um den Gütefaktor abzulesen.
- **Snapping:** `conformal_pec_snap_fraction=0.2` applies in both material variants. The analytical frequency uses the nominal wall position; geometry snapping introduces additional discretization error.
